# Einfache Regression #

In [2]:
import pandas as pd
import statsmodels.api as sm


def regression_aus_excel(
    dateipfad,
    zielvariable,
    einflussvariablen,
    tabellenblatt=0 ):
    """
    Führt eine lineare Regression auf Basis einer Excel-Datei durch.

    Parameter:
    ----------
    dateipfad : str
        Pfad zur Excel-Datei

    zielvariable : str
        Name der Zielvariable (y)

    einflussvariablen : list
        Liste der Einflussvariablen (X)

    tabellenblatt : str oder int
        Name oder Index des Tabellenblatts
    """
    # Nur relevante Spalten auswählen
    relevante_spalten = [zielvariable] + einflussvariablen

    relevant_columns = [
    'id', 'listing_url', 'neighbourhood_cleansed', 'room_type', 'accommodates',
    'price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 
    'review_scores_rating', 'calculated_host_listings_count', 'availability_365',
    'instant_bookable',
    'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location', 'review_scores_value'
    ]
    
    df = pd.read_csv(dateipfad, usecols=relevant_columns)

    # 2. Preis-Bereinigung (von Text zu Zahl)
    df['price'] = df['price'].astype(str).str.replace('$', '').str.replace(',', '')
    df['price'] = pd.to_numeric(df['price'], errors='coerce')



    # Daten bereinigen

    price_95_limit = df['price'].quantile(0.95)

    # B. Kombinierter Filter:

    # 1. Preis muss echt sein (> 0) und unter dem 95. Perzentil liegen
    # 2. Mindestens 3 Reviews (Ausschluss von Karteileichen)
    df_clean = df[
        (df['price'] > 0) & 
        (df['price'] <= price_95_limit) & 
        (df['number_of_reviews'] >= 3)
    ].copy()

# C. Sternebewertung voraussetzen (Sicherstellen, dass keine NaN-Werte für euer 2. Target existieren)
    df = df_clean.dropna(subset=['review_scores_rating'])


    # Alle Spalten bereinigen
    for spalte in relevante_spalten:

        # In String umwandeln
        df[spalte] = df[spalte].astype(str)

        # Typische Preiszeichen entfernen
        df[spalte] = (
            df[spalte]
            .str.replace("€", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )

        # In Zahlen umwandeln
        df[spalte] = pd.to_numeric(df[spalte], errors="coerce")

    # Fehlende Werte entfernen
    df = df.dropna()

    # Zielvariable
    y = df[zielvariable]

    # Einflussvariablen
    X = df[einflussvariablen]

    # Konstante hinzufügen
    X = sm.add_constant(X)

    # Regression berechnen
    modell = sm.OLS(y, X).fit()

    # Ergebnisse anzeigen
    print(modell.summary())

    # Ergebnisse als DataFrame
    ergebnisse = pd.DataFrame({
        "Variable": modell.params.index,
        "Koeffizient": modell.params.values,
        "p-Wert": modell.pvalues.values
    })

    print("\nErgebnisse gespeichert als:")
    print("regression_ergebnisse.xlsx")

    return modell

In [3]:
regression_aus_excel(
    dateipfad="listings (1).csv.gz",
    zielvariable="review_scores_rating",
    einflussvariablen=["review_scores_accuracy"],
    tabellenblatt="Regression, alle Daten"
)

                             OLS Regression Results                             
Dep. Variable:     review_scores_rating   R-squared:                       0.751
Model:                              OLS   Adj. R-squared:                  0.751
Method:                   Least Squares   F-statistic:                 1.822e+04
Date:                  Mon, 18 May 2026   Prob (F-statistic):               0.00
Time:                          14:16:20   Log-Likelihood:                 3598.5
No. Observations:                  6054   AIC:                            -7193.
Df Residuals:                      6052   BIC:                            -7180.
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const   

In [15]:
regression_aus_excel(
    dateipfad="listings (1).csv.gz",
    zielvariable="review_scores_rating",
    einflussvariablen=["review_scores_cleanliness"],
    tabellenblatt="Regression, alle Daten"
)

                             OLS Regression Results                             
Dep. Variable:     review_scores_rating   R-squared:                       0.621
Model:                              OLS   Adj. R-squared:                  0.621
Method:                   Least Squares   F-statistic:                     9909.
Date:                  Mon, 18 May 2026   Prob (F-statistic):               0.00
Time:                          14:12:04   Log-Likelihood:                 2330.2
No. Observations:                  6054   AIC:                            -4656.
Df Residuals:                      6052   BIC:                            -4643.
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
co

In [24]:
regression_aus_excel(
    dateipfad="listings (1).csv.gz",
    zielvariable="review_scores_rating",
    einflussvariablen=["review_scores_checkin"],
    tabellenblatt="Regression, alle Daten"
)

                             OLS Regression Results                             
Dep. Variable:     review_scores_rating   R-squared:                       0.478
Model:                              OLS   Adj. R-squared:                  0.478
Method:                   Least Squares   F-statistic:                     5551.
Date:                  Mon, 18 May 2026   Prob (F-statistic):               0.00
Time:                          14:15:08   Log-Likelihood:                 1365.1
No. Observations:                  6054   AIC:                            -2726.
Df Residuals:                      6052   BIC:                            -2713.
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const     

In [23]:
regression_aus_excel(
    dateipfad="listings (1).csv.gz",
    zielvariable="review_scores_rating",
    einflussvariablen=["review_scores_communication"],
    tabellenblatt="Regression, alle Daten"
)

                             OLS Regression Results                             
Dep. Variable:     review_scores_rating   R-squared:                       0.589
Model:                              OLS   Adj. R-squared:                  0.588
Method:                   Least Squares   F-statistic:                     8656.
Date:                  Mon, 18 May 2026   Prob (F-statistic):               0.00
Time:                          14:15:03   Log-Likelihood:                 2082.9
No. Observations:                  6054   AIC:                            -4162.
Df Residuals:                      6052   BIC:                            -4148.
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------

In [22]:
regression_aus_excel(
    dateipfad="listings (1).csv.gz",
    zielvariable="review_scores_rating",
    einflussvariablen=["review_scores_location"],
    tabellenblatt="Regression, alle Daten"
)

                             OLS Regression Results                             
Dep. Variable:     review_scores_rating   R-squared:                       0.324
Model:                              OLS   Adj. R-squared:                  0.324
Method:                   Least Squares   F-statistic:                     2905.
Date:                  Mon, 18 May 2026   Prob (F-statistic):               0.00
Time:                          14:14:59   Log-Likelihood:                 581.52
No. Observations:                  6054   AIC:                            -1159.
Df Residuals:                      6052   BIC:                            -1146.
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const   

In [21]:
regression_aus_excel(
    dateipfad="listings (1).csv.gz",
    zielvariable="review_scores_rating",
    einflussvariablen=["review_scores_value"],
    tabellenblatt="Regression, alle Daten"
)

                             OLS Regression Results                             
Dep. Variable:     review_scores_rating   R-squared:                       0.735
Model:                              OLS   Adj. R-squared:                  0.735
Method:                   Least Squares   F-statistic:                 1.675e+04
Date:                  Mon, 18 May 2026   Prob (F-statistic):               0.00
Time:                          14:14:55   Log-Likelihood:                 3410.5
No. Observations:                  6054   AIC:                            -6817.
Df Residuals:                      6052   BIC:                            -6804.
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const         

In [20]:
regression_aus_excel(
    dateipfad="listings (1).csv.gz",
    zielvariable="review_scores_rating",
    einflussvariablen=["number_of_reviews"],
    tabellenblatt="Regression, alle Daten"
)

                             OLS Regression Results                             
Dep. Variable:     review_scores_rating   R-squared:                       0.001
Model:                              OLS   Adj. R-squared:                  0.000
Method:                   Least Squares   F-statistic:                     3.976
Date:                  Mon, 18 May 2026   Prob (F-statistic):             0.0462
Time:                          14:14:48   Log-Likelihood:                -603.20
No. Observations:                  6054   AIC:                             1210.
Df Residuals:                      6052   BIC:                             1224.
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const             

In [19]:
regression_aus_excel(
    dateipfad="listings (1).csv.gz",
    zielvariable="review_scores_rating",
    einflussvariablen=["price"],
    tabellenblatt="Regression, alle Daten"
)

                             OLS Regression Results                             
Dep. Variable:     review_scores_rating   R-squared:                       0.000
Model:                              OLS   Adj. R-squared:                 -0.000
Method:                   Least Squares   F-statistic:                    0.5270
Date:                  Mon, 18 May 2026   Prob (F-statistic):              0.468
Time:                          14:14:25   Log-Likelihood:                -604.92
No. Observations:                  6054   AIC:                             1214.
Df Residuals:                      6052   BIC:                             1227.
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.7487      0.008